# Part A: Location Matching - Entity Resolution

## Imports + paths

In [2]:
import sys
from pathlib import Path
import pandas as pd

p = Path.cwd().resolve()
for parent in [p] + list(p.parents):
    if (parent / "src" / "config.py").exists():
        sys.path.insert(0, str(parent))
        break

from src.config import (
    RAW_PATH, LOC_PATH,
    OUT_DIR, FIG_DIR, INT_DIR,
    CAND_ZIP_PATH, MATCH_ZIP_PATH,
    CAND_FALLBACK_TOPK_PATH, MATCH_FALLBACK_PATH,
    FINAL_MATCHES_PATH, FIN_TABLE_PATH, FIN_TABLE_CSV_PATH,
    DUCKDB_THREADS, DUCKDB_MEMORY,
    MATCH_HI, MATCH_LO, GAP_MIN,
    W_NAME, W_ADDR, TOPK
)
from src.duckdb_utils import connect_duckdb
from src.matching import add_standard_cols_raw, add_standard_cols_loc, norm_name, norm_address, score_candidates

print("ROOT OK. outputs:", OUT_DIR)
print("intermediate:", INT_DIR)


ROOT OK. outputs: D:\CenterCheck Assessment\outputs
intermediate: D:\CenterCheck Assessment\outputs\intermediate


## load columns

In [3]:
raw_df = pd.read_parquet(RAW_PATH, columns=[
    "id","name","address","city","state","postal_code","start_at","end_at","revenue"
])
loc_df = pd.read_parquet(LOC_PATH, columns=[
    "id","business_entity_id","name","street_address","city","state","postal_code","area_sq_ft"
]).rename(columns={"id":"location_id"})

## Standardize

In [4]:
raw_df = add_standard_cols_raw(raw_df)
loc_df = add_standard_cols_loc(loc_df)

print("raw zip5 empty rate:", (raw_df["zip5"]=="").mean())
print("loc zip5 empty rate:", (loc_df["zip5"]=="").mean())

raw zip5 empty rate: 0.0
loc zip5 empty rate: 0.0


## Pass 1 candidates (state+zip5) 

In [5]:
con = connect_duckdb(DUCKDB_THREADS, DUCKDB_MEMORY)
con.register("raw_df", raw_df[["id","name_norm","addr_norm","state_norm","zip5","block_zip"]])
con.register("loc_df", loc_df[["location_id","business_entity_id","area_sq_ft","name_norm","addr_norm","state_norm","zip5","block_zip"]])

con.execute(f"""
COPY (
  SELECT
    r.id AS raw_id,
    l.location_id,
    r.name_norm AS raw_name_norm,
    r.addr_norm AS raw_addr_norm,
    l.name_norm AS loc_name_norm,
    l.addr_norm AS loc_addr_norm,
    l.business_entity_id,
    l.area_sq_ft
  FROM raw_df r
  JOIN loc_df l
    ON r.block_zip = l.block_zip
) TO '{CAND_ZIP_PATH.as_posix()}' (FORMAT PARQUET);
""")
print("saved:", CAND_ZIP_PATH)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

saved: D:\CenterCheck Assessment\outputs\intermediate\candidates_zip.parquet


## score pass 1 + best match

In [6]:
cand = pd.read_parquet(CAND_ZIP_PATH)
cand = score_candidates(cand, w_name=W_NAME, w_addr=W_ADDR)

cand_small = cand[["raw_id","location_id","final_score","business_entity_id","area_sq_ft"]].copy()
del cand

con = connect_duckdb(DUCKDB_THREADS, DUCKDB_MEMORY)
con.register("cand", cand_small)

matches_zip = con.execute(f"""
WITH ranked AS (
  SELECT
    raw_id, location_id, final_score, business_entity_id, area_sq_ft,
    ROW_NUMBER() OVER (PARTITION BY raw_id ORDER BY final_score DESC) AS rn,
    LEAD(final_score) OVER (PARTITION BY raw_id ORDER BY final_score DESC) AS second_score
  FROM cand
),
best AS (
  SELECT
    raw_id,
    location_id,
    final_score,
    second_score,
    CASE WHEN second_score IS NULL THEN NULL ELSE final_score - second_score END AS gap,
    business_entity_id,
    area_sq_ft,
    'zip5_pass' AS match_source
  FROM ranked
  WHERE rn = 1
)
SELECT
  *,
  CASE
    WHEN final_score >= {MATCH_HI} AND (second_score IS NULL OR gap >= {GAP_MIN}) THEN 'match'
    WHEN final_score >= {MATCH_LO} THEN 'possible'
    ELSE 'no_match'
  END AS match_status
FROM best
""").df()

matches_zip.to_parquet(MATCH_ZIP_PATH, index=False)
print(matches_zip["match_status"].value_counts())
print("saved:", MATCH_ZIP_PATH)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

match_status
match       4504136
possible      95096
no_match      25114
Name: count, dtype: int64
saved: D:\CenterCheck Assessment\outputs\intermediate\matches_zip.parquet


## Pass 2 Top‑K fallback candidates (state+city+zip3)

In [7]:
zip_raw_ids = set(matches_zip["raw_id"].unique())
leftovers = raw_df[~raw_df["id"].isin(zip_raw_ids)][["id","name","address","state_norm","city_norm","zip3","block_fallback"]].copy()

loc_fb = loc_df[["location_id","business_entity_id","area_sq_ft","name","street_address","state_norm","city_norm","zip3","block_fallback"]].copy()

con = connect_duckdb(threads=2, memory_limit="3GB")
con.register("leftovers", leftovers)
con.register("loc_fb", loc_fb)

con.execute(f"""
COPY (
  WITH joined AS (
    SELECT
      l.id as raw_id,
      b.location_id,
      l.name as raw_name,
      l.address as raw_addr,
      b.name as loc_name,
      b.street_address as loc_addr,
      b.business_entity_id,
      b.area_sq_ft,
      regexp_extract(l.address, '([0-9]+)', 1) as raw_num,
      regexp_extract(b.street_address, '([0-9]+)', 1) as loc_num
    FROM leftovers l
    JOIN loc_fb b
      ON l.block_fallback = b.block_fallback
  ),
  ranked AS (
    SELECT
      *,
      CASE WHEN raw_num != '' AND raw_num = loc_num THEN 1 ELSE 0 END AS num_match,
      abs(length(raw_addr) - length(loc_addr)) AS addr_len_diff,
      abs(length(raw_name) - length(loc_name)) AS name_len_diff,
      row_number() OVER (
        PARTITION BY raw_id
        ORDER BY num_match DESC, addr_len_diff ASC, name_len_diff ASC
      ) AS rn
    FROM joined
  )
  SELECT * EXCLUDE(rn, num_match, addr_len_diff, name_len_diff, raw_num, loc_num)
  FROM ranked
  WHERE rn <= {TOPK}
) TO '{CAND_FALLBACK_TOPK_PATH.as_posix()}' (FORMAT PARQUET);
""")
print("saved:", CAND_FALLBACK_TOPK_PATH)

saved: D:\CenterCheck Assessment\outputs\intermediate\cand_fallback_topk.parquet


## score fallback + best match

In [8]:
cand_fb = pd.read_parquet(CAND_FALLBACK_TOPK_PATH)

cand_fb["raw_name_norm"] = cand_fb["raw_name"].map(norm_name)
cand_fb["raw_addr_norm"] = cand_fb["raw_addr"].map(norm_address)
cand_fb["loc_name_norm"] = cand_fb["loc_name"].map(norm_name)
cand_fb["loc_addr_norm"] = cand_fb["loc_addr"].map(norm_address)

cand_fb = score_candidates(cand_fb, w_name=W_NAME, w_addr=W_ADDR)

cand_fb_small = cand_fb[["raw_id","location_id","final_score","business_entity_id","area_sq_ft"]].copy()
del cand_fb

con = connect_duckdb(threads=2, memory_limit="3GB")
con.register("cand", cand_fb_small)

matches_fb = con.execute(f"""
WITH ranked AS (
  SELECT
    raw_id, location_id, final_score, business_entity_id, area_sq_ft,
    ROW_NUMBER() OVER (PARTITION BY raw_id ORDER BY final_score DESC) AS rn,
    LEAD(final_score) OVER (PARTITION BY raw_id ORDER BY final_score DESC) AS second_score
  FROM cand
),
best AS (
  SELECT
    raw_id,
    location_id,
    final_score,
    second_score,
    CASE WHEN second_score IS NULL THEN NULL ELSE final_score - second_score END AS gap,
    business_entity_id,
    area_sq_ft,
    'fallback_zip3_topk' AS match_source
  FROM ranked
  WHERE rn = 1
)
SELECT
  *,
  CASE
    WHEN final_score >= {MATCH_HI} AND (second_score IS NULL OR gap >= {GAP_MIN}) THEN 'match'
    WHEN final_score >= {MATCH_LO} THEN 'possible'
    ELSE 'no_match'
  END AS match_status
FROM best
""").df()

matches_fb.to_parquet(MATCH_FALLBACK_PATH, index=False)
print(matches_fb["match_status"].value_counts())
print("saved:", MATCH_FALLBACK_PATH)


match_status
match       196409
possible      4207
no_match       410
Name: count, dtype: int64
saved: D:\CenterCheck Assessment\outputs\intermediate\matches_fallback.parquet


## FINAL_MATCHES_PATH (keep all statuses; prefer zip if tie)

In [9]:
final_matches = pd.concat([matches_zip, matches_fb], ignore_index=True)

final_matches["source_rank"] = final_matches["match_source"].map(
    {"zip5_pass": 0, "fallback_zip3_topk": 1}
).fillna(9)

final_matches = final_matches.sort_values(
    ["raw_id","final_score","source_rank"],
    ascending=[True, False, True]
).drop_duplicates("raw_id", keep="first").drop(columns=["source_rank"])

final_matches.to_parquet(FINAL_MATCHES_PATH, index=False)
print(final_matches["match_status"].value_counts())
print(final_matches["match_source"].value_counts())
print("saved:", FINAL_MATCHES_PATH)

match_status
match       4700545
possible      99303
no_match      25524
Name: count, dtype: int64
match_source
zip5_pass             4624346
fallback_zip3_topk     201026
Name: count, dtype: int64
saved: D:\CenterCheck Assessment\outputs\intermediate\final_matches.parquet


## Save CSV/Parquet financial_table

In [10]:
con = connect_duckdb(DUCKDB_THREADS, DUCKDB_MEMORY)

con.execute(f"""
COPY (
  SELECT
    rf.id AS raw_id,

    -- analysis match id
    CASE
      WHEN fm.match_status IN ('match','possible') THEN fm.location_id
      ELSE NULL
    END AS matched_location_id,

    CASE
      WHEN fm.match_status IN ('match','possible') THEN bl.business_entity_id
      ELSE NULL
    END AS business_entity_id,

    CASE
      WHEN fm.match_status IN ('match','possible') THEN bl.area_sq_ft
      ELSE NULL
    END AS area_sq_ft,

    rf.name,
    rf.address,
    rf.city,
    rf.state,
    rf.postal_code,

    CAST(rf.start_at AS DATE) AS start_date,
    CAST(rf.end_at   AS DATE) AS end_date,

    rf.revenue AS raw_card_revenue,
    date_diff('day', CAST(rf.start_at AS DATE), CAST(rf.end_at AS DATE)) + 1 AS days_in_window,
    rf.revenue / NULLIF(date_diff('day', CAST(rf.start_at AS DATE), CAST(rf.end_at AS DATE)) + 1, 0) AS card_rev_per_day,

    COALESCE(fm.match_status, 'unmatched') AS match_status,

    -- diagnostics included in same file
    fm.location_id AS candidate_location_id,
    fm.match_source,
    fm.final_score,
    fm.second_score,
    fm.gap

  FROM read_parquet('{RAW_PATH.as_posix()}') rf
  LEFT JOIN read_parquet('{FINAL_MATCHES_PATH.as_posix()}') fm
    ON rf.id = fm.raw_id
  LEFT JOIN read_parquet('{LOC_PATH.as_posix()}') bl
    ON fm.location_id = bl.id
) TO '{FIN_TABLE_PATH.as_posix()}' (FORMAT PARQUET);
""")

con.execute(f"""
COPY (
  SELECT * FROM read_parquet('{FIN_TABLE_PATH.as_posix()}')
) TO '{FIN_TABLE_CSV_PATH.as_posix()}'
(HEADER, DELIMITER ',');
""")

print("saved parquet:", FIN_TABLE_PATH)
print("saved CSV:", FIN_TABLE_CSV_PATH)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

saved parquet: D:\CenterCheck Assessment\outputs\financial_table.parquet
saved CSV: D:\CenterCheck Assessment\outputs\financial_table.csv


## Sanity checks

In [11]:
from src.sanity_checks import run_part_a_checks
coverage, status, uniq = run_part_a_checks(FIN_TABLE_PATH, FINAL_MATCHES_PATH)
display(coverage)
display(status)
display(uniq)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_raw_rows,matched_rows,unmatched_rows,matched_pct
0,4825372,4799848.0,25524.0,99.47


,match_status,n,pct
0,match,4700545,97.41
1,possible,99303,2.06
2,no_match,25524,0.53


,match_rows,distinct_raw_id,duplicate_raw_id_rows
0,4825372,4825372,0
